### Restart and Run All Cells

In [2]:
import pandas as pd
from datetime import date, timedelta, datetime
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///c:\\ruby\\portlt\\db\\development.sqlite3")
conlt = engine.connect()

year = 2026
quarter = 1
current_time = datetime.now()
formatted_time = current_time.strftime("%Y-%m-%d %H:%M:%S")
print(formatted_time )

2026-05-17 14:52:01


In [3]:
cols = 'name year quarter q_amt_c q_amt_p inc_profit percent'.split()

format_dict = {
                'q_amt':'{:,}','q_amt_c':'{:,}','q_amt_p':'{:,}','inc_profit':'{:,}',
                'yoy_gain':'{:,}','acc_gain':'{:,}',   
                'latest_amt':'{:,}','previous_amt':'{:,}','inc_amt':'{:,}',
                'q_eps':'{:.4f}','y_eps':'{:.4f}','aq_eps':'{:.4f}','ay_eps':'{:.4f}',
                'percent':'{:.2f}%','inc_pct':'{:.2f}%'
              }

In [4]:
sql = """
SELECT name,year,quarter,q_amt 
FROM epss 
WHERE (year = %s AND quarter <= %s) 
OR (year = %s-1 AND quarter >= %s+1)
ORDER BY year DESC, quarter DESC
"""
sql = sql % (year, quarter, year, quarter)
print(sql)


SELECT name,year,quarter,q_amt 
FROM epss 
WHERE (year = 2026 AND quarter <= 1) 
OR (year = 2026-1 AND quarter >= 1+1)
ORDER BY year DESC, quarter DESC



In [5]:
dfc = pd.read_sql(sql, conlt)
dfc["Counter"] = 1
dfc_grp = dfc.groupby(["name"], as_index=False).sum()
dfc_grp = dfc_grp[dfc_grp["Counter"] == 4]
dfc_grp.shape

(181, 5)

In [6]:
sql = """
SELECT name,year,quarter,q_amt 
FROM epss 
WHERE (year = %s AND quarter <= %s-1) 
OR (year = %s-1 AND quarter >= %s) 
ORDER BY year DESC, quarter DESC"""
sql = sql % (year, quarter, year, quarter)
print(sql)


SELECT name,year,quarter,q_amt 
FROM epss 
WHERE (year = 2026 AND quarter <= 1-1) 
OR (year = 2026-1 AND quarter >= 1) 
ORDER BY year DESC, quarter DESC


In [7]:
dfp = pd.read_sql(sql, conlt)
dfp["Counter"] = 1
dfp_grp = dfp.groupby(["name"], as_index=False).sum()
dfp_grp = dfp_grp[dfp_grp["Counter"] == 4]
dfp_grp.head().style.format(format_dict)

,name,year,quarter,q_amt,Counter
0,3BBIF,8100,10,"10,827,771",4
1,ACE,8100,10,"798,614",4
2,ADVANC,8100,10,"47,885,902",4
3,AEONTS,8100,10,"3,094,125",4
4,AH,8100,10,"731,428",4


In [8]:
dfp.name.unique().shape

(197,)

In [9]:
dfm = pd.merge(dfc_grp, dfp_grp, on="name", suffixes=(["_c", "_p"]), how="inner")
dfm["inc_profit"] = dfm["q_amt_c"] - dfm["q_amt_p"]
dfm["percent"] = round(dfm["inc_profit"] / abs(dfm["q_amt_p"]) * 100, 2)
dfm["year"] = year
dfm["quarter"] = "Q" + str(quarter)
df_percent = dfm[cols]
df_percent.head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2026,Q1,"6,831,513","10,827,771","-3,996,258",-36.91%
1,ACE,2026,Q1,"886,861","798,614","88,247",11.05%
2,ADVANC,2026,Q1,"50,797,881","47,885,902","2,911,979",6.08%
3,AH,2026,Q1,"739,606","731,428","8,178",1.12%
4,AIE,2026,Q1,"62,999","21,904","41,095",187.61%


In [10]:
# Create the SQL query with parameter binding
sql = text("DELETE FROM qt_profits WHERE year = :year AND quarter = :quarter")

# Execute the query with parameters
params = {'year': year, 'quarter': f'Q{quarter}'}
rp = conlt.execute(sql, params)

# Print the number of rows affected
print("Rows deleted:", rp.rowcount)

Rows deleted: 54


In [11]:
sql = "SELECT name, id FROM tickers"
tickers = pd.read_sql(sql, conlt)
tickers.sample(5)

,name,id
200,NCH,317
131,HMPRO,208
57,BLA,70
9,AIE,720
129,HANA,205


In [12]:
df_ins = pd.merge(df_percent, tickers, on="name", how="inner")
df_ins.head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent,id
0,3BBIF,2026,Q1,"6,831,513","10,827,771","-3,996,258",-36.91%,234
1,ACE,2026,Q1,"886,861","798,614","88,247",11.05%,698
2,ADVANC,2026,Q1,"50,797,881","47,885,902","2,911,979",6.08%,6
3,AH,2026,Q1,"739,606","731,428","8,178",1.12%,9
4,AIE,2026,Q1,"62,999","21,904","41,095",187.61%,720


In [13]:
# Convert DataFrame to list of records
rcds = df_ins.values.tolist()

# Define column names in the same order as values
columns = ['name', 'year', 'quarter', 'latest_amt', 'previous_amt', 'inc_amt', 'inc_pct', 'ticker_id']

# SQL insert statement with named parameters
sql = text("""
    INSERT INTO qt_profits 
    (name, year, quarter, latest_amt, previous_amt, inc_amt, inc_pct, ticker_id)
    VALUES (:name, :year, :quarter, :latest_amt, :previous_amt, :inc_amt, :inc_pct, :ticker_id)
""")

try:
    # Execute inserts
    for rcd in rcds:
        # Convert list to dictionary
        params = dict(zip(columns, rcd))
        conlt.execute(sql, params)
except Exception as e:
    raise e

### End of loop

In [15]:
criteria_1 = df_ins.q_amt_c > 440_000
df_ins.loc[criteria_1, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2026,Q1,"6,831,513","10,827,771","-3,996,258",-36.91%
1,ACE,2026,Q1,"886,861","798,614","88,247",11.05%
2,ADVANC,2026,Q1,"50,797,881","47,885,902","2,911,979",6.08%
3,AH,2026,Q1,"739,606","731,428","8,178",1.12%
5,AIMIRT,2026,Q1,"702,550","684,130","18,420",2.69%


In [16]:
criteria_2 = df_ins.q_amt_p > 400_000
df_ins.loc[criteria_2, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2026,Q1,"6,831,513","10,827,771","-3,996,258",-36.91%
1,ACE,2026,Q1,"886,861","798,614","88,247",11.05%
2,ADVANC,2026,Q1,"50,797,881","47,885,902","2,911,979",6.08%
3,AH,2026,Q1,"739,606","731,428","8,178",1.12%
5,AIMIRT,2026,Q1,"702,550","684,130","18,420",2.69%


In [17]:
criteria_3 = df_ins.percent > 10.00
df_ins.loc[criteria_3, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
1,ACE,2026,Q1,"886,861","798,614","88,247",11.05%
4,AIE,2026,Q1,"62,999","21,904","41,095",187.61%
7,AMATA,2026,Q1,"3,697,994","3,148,658","549,336",17.45%
8,ANAN,2026,Q1,"187,831","2,715","185,116",6818.27%
12,ASK,2026,Q1,"587,609","531,545","56,064",10.55%


In [18]:
df_ins_criteria = criteria_1 & criteria_2 & criteria_3
df_ins.loc[df_ins_criteria, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
1,ACE,2026,Q1,"886,861","798,614","88,247",11.05%
7,AMATA,2026,Q1,"3,697,994","3,148,658","549,336",17.45%
12,ASK,2026,Q1,"587,609","531,545","56,064",10.55%
16,BA,2026,Q1,"3,965,351","3,549,206","416,145",11.73%
21,BCP,2026,Q1,"6,908,168","2,879,701","4,028,467",139.89%


In [19]:
df_ins[df_ins_criteria].sort_values(by=["percent"], ascending=[False]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent,id
139,SPRC,2026,Q1,"9,222,759","2,569,354","6,653,405",258.95%,470
33,BPP,2026,Q1,"8,329,005","3,026,277","5,302,728",175.22%,74
21,BCP,2026,Q1,"6,908,168","2,879,701","4,028,467",139.89%,52
159,TOP,2026,Q1,"30,561,848","14,584,201","15,977,647",109.55%,555
35,CENTEL,2026,Q1,"3,387,598","1,992,901","1,394,697",69.98%,92


In [20]:
df_ins[df_ins_criteria].sort_values(by=["name"], ascending=[True]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent,id
1,ACE,2026,Q1,"886,861","798,614","88,247",11.05%,698
7,AMATA,2026,Q1,"3,697,994","3,148,658","549,336",17.45%,21
12,ASK,2026,Q1,"587,609","531,545","56,064",10.55%,38
16,BA,2026,Q1,"3,965,351","3,549,206","416,145",11.73%,45
21,BCP,2026,Q1,"6,908,168","2,879,701","4,028,467",139.89%,52


In [21]:
df_ins[df_ins_criteria].sort_values(by=["name"], ascending=[True]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent,id
1,ACE,2026,Q1,"886,861","798,614","88,247",11.05%,698
7,AMATA,2026,Q1,"3,697,994","3,148,658","549,336",17.45%,21
12,ASK,2026,Q1,"587,609","531,545","56,064",10.55%,38
16,BA,2026,Q1,"3,965,351","3,549,206","416,145",11.73%,45
21,BCP,2026,Q1,"6,908,168","2,879,701","4,028,467",139.89%,52


In [22]:
conlt.commit()
conlt.close()

In [23]:
current_time = datetime.now()
formatted_time = current_time.strftime("%Y:%m:%d %H:%M:%S")
print(formatted_time)

2026:05:17 14:52:02
